# Exploración de PDB - familia proteíca "Isomerasas" 

## Objetivo
Desarrollar un análisis exploratorio de las estructuras de una familia de proteínas depositadas en el PDB,
utilizando consultas a su API y herramientas básicas de programación y visualización.

### Parte 1: Consulta de estructuras en el PDB



In [1]:
# Definición de variables y cargar librerias
import requests

url = "https://search.rcsb.org/rcsbsearch/v2/query"
pfam_ID = "PF00697" #código de la familia de proteinas
n_familia = "phosphoribosylanthranilate isomerase" #nombre de la familia

In [2]:
# Consulta - estamos buscando todas las estructuras de PDB que contengan
# una proteína de la familia PRAI con el código pfam
def consulta_pdb(query_node, return_type="entry"):
    payload = {
        "query": query_node,
        "return_type": return_type,
        "request_options": {"return_all_hits": True},
    }
    r = requests.post(url, json=payload, timeout=60)
    if r.status_code == 204:
        return []
    r.raise_for_status()
    data = r.json()
    print(f"total_count reportado por el API: {data.get('total_count')}") # numero de resultados por el API
    return [hit["identifier"] for hit in data["result_set"]]

In [3]:
# Busqueda por pfam
query_pfam = {
    "type": "group",
    "logical_operator": "and",
    "nodes": [
        {"type": "terminal", "service": "text",
         # primera condicion - que el tipo sea pfam
         "parameters": {"attribute": "rcsb_polymer_entity_annotation.type",
                        "operator": "exact_match", "value": "Pfam"}}, 
        {"type": "terminal", "service": "text",
         # segunda condicion - que el codigo de anotacion sea el que pfam_ID
         "parameters": {"attribute": "rcsb_polymer_entity_annotation.annotation_id",
                        "operator": "exact_match", "value": pfam_ID}},
    ],
}

ids_pfam = consulta_pdb(query_pfam) #se consulta y guarda la lista de IDs
print(f"Entradas por pfam ({pfam_ID}): {len(ids_pfam)}")


total_count reportado por el API: 11
Entradas por pfam (PF00697): 11


In [4]:
# Busqueda por familia
# Se busca el nombre de la familia en cualquier texto y es más amplia que pfam
query_texto = {
    "type": "terminal",
    "service": "full_text",
    "parameters": {"value": n_familia},
}
ids_texto = consulta_pdb(query_texto)
print(f"Entradas por texto '{n_familia}': {len(ids_texto)}")

total_count reportado por el API: 34
Entradas por texto 'phosphoribosylanthranilate isomerase': 34


In [5]:
# Comparación del numero reportado por la busqueda por pfam y por el nombre de la familia
set_pfam, set_texto = set(ids_pfam), set(ids_texto)
print("Resultados por pfam (IDs):", len(set_pfam - set_texto))
print("Resultados por busqueda de texto (IDs):", len(set_texto - set_pfam))
print("Cuales estan en ambos:", len(set_pfam & set_texto))

ids_final = sorted(set_pfam) # guardamos las de pfam por que a veces el nombre no encuentra
# todas las estructuras y en muchos casos suelen ser falsos positivos
with open("ids_familia.txt", "w") as f:
    f.write("\n".join(ids_final))
print(f"Guardados {len(ids_final)} IDs en ids_familia.txt") # guardamos los IDs

Resultados por pfam (IDs): 0
Resultados por busqueda de texto (IDs): 23
Cuales estan en ambos: 11
Guardados 11 IDs en ids_familia.txt
